# 03 — Gloss drafts (G0 v0.1) + validation (Day 2 Step 2)

Runs exactly the same code as `uv run python -m bdsl.cli gloss-draft --data pilot50` and `… gloss-validate` (see `CLAUDE.md` → COMMANDS).

- **What G0 writes:** `auto_draft` pseudo-glosses only (ADR-0024). It never reorders words and never guesses a lemma (`?`); `~` marks a suggested genitive stem.
- **Human work is protected:** `gloss-draft` refuses to overwrite `annotations/pilot50_v1_gloss_v01.csv` once any row has been reviewed.
- **Reviews:** only a team member registered by the owner in `annotations/REVIEWERS.csv` sets `team_reviewed`; only a registered expert sets `expert_validated`. The AI never does (ADR-0015). How to review: `annotations/REVIEW_GUIDE.md`.
- **Public repo:** `annotations/` is tracked, so a push publishes it. Reviewer names never go in it, only codes.

In [ ]:
from bdsl.cli import main
from bdsl.config import load_config

cfg = load_config(data="pilot50")
print("env:", cfg["_meta"]["env"], "| annotations:", cfg["paths"]["annotations_dir"])

In [ ]:
# G0 drafts (exit code 2 = refused because reviewed rows exist, which is the intended protection).
print("exit code:", main(["gloss-draft", "--data", "pilot50"]))

In [ ]:
# Validation: schema, statuses, registered reviewers, no mock rows. Run after every review session.
print("exit code:", main(["gloss-validate"]))

In [ ]:
# A look at the drafts and at which rows a model may learn from.
from pathlib import Path

from bdsl.text.annotations import read_csv, usable_for_training

df = read_csv(Path(cfg["paths"]["annotations_dir"]) / cfg["annotations"]["pilot_file"])
print(df["status"].value_counts().to_dict(), "| usable for training:", len(usable_for_training(df)))
df[["index", "split", "sentence_norm", "gloss_auto", "tags_auto", "status"]].head(10)